#### # 1. Data Ingestion & Base Staging Setup

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# Extract data from Bronze staging layer
bronze_df = spark.table("workspace.bronze.erp_cust_az12")

# Staging step to maintain structural architecture consistency 
standardized_df = bronze_df

#### # 2. Business Rules & Customer Cleansing Logic

In [0]:
# Apply business rule logic for customer IDs, future birthdates, and gender normalization
transformed_df = (
    standardized_df
    # 1. Identifiers: Remove 'NAS' prefix from customer ID if present
    .withColumn(
        "cid",
        F.when(F.col("cid").like("NAS%"), F.substring(F.col("cid"), 4, F.length(F.col("cid"))))
        .otherwise(F.col("cid"))
    )
    # 2. Attributes (Dates): Replace future birthdates with NULL
    .withColumn(
        "bdate",
        F.when(F.col("bdate") > F.current_date(), F.lit(None))
        .otherwise(F.col("bdate"))
    )
    # 3. Attributes (Categorical): Standardize and normalize gender values
    .withColumn(
        "gen",
        F.when(F.upper(F.trim(F.col("gen"))).isin("F", "FEMALE"), F.lit("Female"))
        .when(F.upper(F.trim(F.col("gen"))).isin("M", "MALE"), F.lit("Male"))
        .otherwise(F.lit("N/A"))
    )
    # 4. Audit Metadata: Operational tracking timestamp
    .withColumn("dwh_create_date", F.current_timestamp())
)

#### # 3. Final Schema Formatting, Renaming, and Target Storage

In [0]:
# Grouping DDL casting and structural organization together (Ordered Sequence)
final_df = transformed_df.select(
    F.col("cid").cast("string"),            # Primary Key / Identifier
    F.col("bdate").cast("date"),            # Demographics - Birth Date
    F.col("gen").cast("string"),            # Demographics - Gender
    F.col("dwh_create_date")                # Warehouse Audit Metadata
)

# Reference mapping ordered exactly to match the selection sequence above
RENAME_MAP = {
    "cid": "customer_id",
    "bdate": "birth_date",
    "gen": "gender"
}

renamed_df = final_df
for old_name, new_name in RENAME_MAP.items():
    renamed_df = renamed_df.withColumnRenamed(old_name, new_name)

# Write output schema directly to Silver Delta layer
renamed_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.erp_customer")

# Display organized sample preview rows interactively
renamed_df.limit(10).display()